## tl;dr

At 260 units, $2,180 monthly rent, 95% collectible rent, and 80% routing, annual PPV is $5,169,216. A 10 bps net take yields $5,169 per asset, or $1.66 per total unit per month, and requires about 50,298 comparable units for $1M of annual net payment revenue. Retention assumptions dominate lifetime economics, so the decision metrics are realized dollars per retained unit per year and payment-engine survival at stabilization—not bps alone.

Execution note: the workspace has Python 3.9 but not `jupyter`, `nbformat`, or `nbclient`. The same code cells were run directly with Python 3 on 2026-09-02 and the outputs were reconciled into the canonical Overseer memo. To execute this notebook locally, install Jupyter and run `python -m jupyter nbconvert --execute --to notebook --inplace spec/PAYMENT_ENGINE_ECONOMICS_VALIDATION.ipynb`.

## Context & Methods

This is a deterministic validation companion for `spec/OVERSEER_MODEL_AND_AGENT_SYNTHESIS.md`. Repository inputs are 260 units and $2,180 average monthly rent. The 95% collectible share and 80% LeaseRight routing rate are explicit underwriting assumptions, not observed sponsor data. Retention scenarios are stress tests, not market estimates.

Primary external source inputs: AppFolio 2025 Form 10-K reports $721.549M of Value Added Services revenue and 9.4M ending units. The filing records payment fees gross of payment-processing costs and combines payments with other services, so the per-unit conversion is only a scale check.

In [ ]:
units = 260
monthly_rent = 2_180
collectible_share = 0.95
routing_share = 0.80
annual_ppv = units * monthly_rent * 12 * collectible_share * routing_share
ppv_per_unit = annual_ppv / units
annual_ppv, ppv_per_unit

## Results

### Take-rate sensitivity

In [ ]:
take_rate_results = []
for bps in (5, 10, 25):
    annual_asset_revenue = annual_ppv * bps / 10_000
    take_rate_results.append({
        'net_bps': bps,
        'annual_asset_revenue': round(annual_asset_revenue, 2),
        'revenue_per_unit_month': round(annual_asset_revenue / units / 12, 4),
        'ppv_for_1m_revenue': round(1_000_000 / (bps / 10_000), 2),
        'units_for_1m_revenue': round(1_000_000 / (ppv_per_unit * bps / 10_000)),
    })
take_rate_results

### AppFolio scale check

In [ ]:
appfolio_vas_revenue = 721_549_000
appfolio_ending_units = 9_400_000
appfolio_vas_per_ending_unit = appfolio_vas_revenue / appfolio_ending_units
appfolio_bps_equivalent = appfolio_vas_per_ending_unit / ppv_per_unit * 10_000
round(appfolio_vas_per_ending_unit, 2), round(appfolio_bps_equivalent, 1)

### Retention stress test

`survival` is the share of won lease-up assets whose payment relationship survives stabilization. `annual_attrition` applies to the surviving base. Lifetime figures are undiscounted.

In [ ]:
annual_revenue_at_10bps = annual_ppv * 10 / 10_000
target_retained_units = 1_000_000 / (ppv_per_unit * 10 / 10_000)
retention_results = []
for survival, annual_attrition in ((0.35, 0.15), (0.60, 0.12), (1.00, 0.15)):
    lifetime_revenue = annual_revenue_at_10bps * survival / annual_attrition
    retained_units_per_annual_win = units * survival / annual_attrition
    annual_wins = target_retained_units / retained_units_per_annual_win
    crossover_bps = 25_000 / (annual_ppv * survival / annual_attrition) * 10_000
    retention_results.append({
        'survival': survival,
        'annual_attrition': annual_attrition,
        'lifetime_revenue_at_10bps': round(lifetime_revenue),
        'annual_260_unit_wins_for_1m': round(annual_wins, 1),
        'bps_to_equal_25k_project_fee': round(crossover_bps, 1),
    })
retention_results

## Takeaways

- Five to ten net bps is a defensible commercial screen, not an observed LeaseRight margin.
- Ten bps is $1.66 per total unit per month and needs about 50K comparable units for $1M annual revenue.
- A 25 bps case remains a stretch until a written partner proposal establishes the fee base and all variable costs.
- Fifty bps is intentionally omitted from the plan.
- The retention scenarios show why contract survival and a lower-CAC stabilized-unit acquisition path can matter more than incremental processor basis points.